In [504]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, MaxAbsScaler

from sklearn.metrics import accuracy_score, classification_report

from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

# Classification Exercise
## Building complex models, data pipelines, and reproducibility

In this exercise, we'll be using the [Spaceship Titanic](https://www.kaggle.com/competitions/spaceship-titanic/) dataset from Kaggle.

Take some time to look at the leaderboard, maybe even look at a few submissions to get ideas. Keep in mind, however, that most people that enter this particular competition are complete beginners. It's **very** likely that their approach is wrong, badly written, messy, or not reproducible. Don't rush to copy everything you see; apply your critical thinking skills even to projects which score really well on the leaderboard. This is, after all, just a simple score on a small testing set; not something that would go into a production environment.

### Problem 1. Load and inspect the data
The goal / broad idea of this analysis is to model _which passengers were transported_. Obviously, this is a binary classification task.

Analyze the data to get a better idea what it contains. Don't forget to look at the metadata as well - this is the Kaggle description of the dataset.

Pay attention to `PassengerId` and `Cabin` because they look like IDs but contain useful info. 

\* Optionally, have a special look at the `Name` column and decide what to do with it.

Some questions you might want to answer are:
* What are the data types? Also think about value ranges because you'll have to rescale the columns later.
* Is the dataset balanced?
* What can we do about missing values?
* How to extract info from the IDs?
* Are the features correlated in some way (and is this a good, or a bad thing)?

but don't feel constrained to only these.

You should remember that we define the experimental procedure **BEFORE** seeing the actual data. There's no need for null hypotheses, the models are already hypotheses. But try to fill in the following list **AND DO YOUR BEST TO NOT CHANGE THE PROCEDURE LATER**!

* Validation method (e.g., test set statistics) 
* Main metric (e.g., accuracy - describe what you're choosing and why)
* Secondary metrics (e.g. other confusion matrix metrics, but also maybe inference time, max number of features, etc. I'd try to add at least one graphical / dataviz metric too)
* Baseline model: you may want to put logistic regression here, but we're actually also going to train a dummy model
* Rules for comparing models: how to measure _significant_ differences in score. Don't confuse the validation and testing sets!
* Rules for adding features: how to decide if the model needs more features (e.g., proven high bias on validation set)

Try to be careful about your experimental procedure and don't forget to document everything. Pay a lot of attention into _why_ you're using one metric, score, rule, idea; if possible - comment on why you aren't using other popular ones.

In [505]:
data = pd.read_csv('data/train.csv')
data

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,9276_01,Europa,False,A/98/P,55 Cancri e,41.0,True,0.0,6819.0,0.0,1643.0,74.0,Gravior Noxnuther,False
8689,9278_01,Earth,True,G/1499/S,PSO J318.5-22,18.0,False,0.0,0.0,0.0,0.0,0.0,Kurta Mondalley,False
8690,9279_01,Earth,False,G/1500/S,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,1.0,0.0,Fayey Connon,True
8691,9280_01,Europa,False,E/608/S,55 Cancri e,32.0,False,0.0,1049.0,0.0,353.0,3235.0,Celeon Hontichre,False


In [506]:
data_pipeline_preprocess = data.copy()

In [507]:
data_pipeline_preprocess

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,9276_01,Europa,False,A/98/P,55 Cancri e,41.0,True,0.0,6819.0,0.0,1643.0,74.0,Gravior Noxnuther,False
8689,9278_01,Earth,True,G/1499/S,PSO J318.5-22,18.0,False,0.0,0.0,0.0,0.0,0.0,Kurta Mondalley,False
8690,9279_01,Earth,False,G/1500/S,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,1.0,0.0,Fayey Connon,True
8691,9280_01,Europa,False,E/608/S,55 Cancri e,32.0,False,0.0,1049.0,0.0,353.0,3235.0,Celeon Hontichre,False


In [508]:
data_pipeline_preprocess = data_pipeline_preprocess.drop(columns=['PassengerId'])

In [509]:
data_pipeline_preprocess['total_spend'] = data_pipeline_preprocess['RoomService'] + \
    data_pipeline_preprocess['FoodCourt'] + data_pipeline_preprocess['ShoppingMall'] + data_pipeline_preprocess['Spa'] + \
    data_pipeline_preprocess['VRDeck']

In [510]:
data_pipeline_preprocess = data_pipeline_preprocess.drop(columns=['Name'])
data_pipeline_preprocess[['Deck', 'CabinNum', 'Side']] = (data_pipeline_preprocess['Cabin'].str.split('/', expand=True))
data_pipeline_preprocess = data_pipeline_preprocess.drop(columns=['Cabin'])

In [511]:
attributes, target = data_pipeline_preprocess.drop(columns=['Transported']), data_pipeline_preprocess['Transported']

In [512]:
attributes_train, attributes_test, target_train, target_test = train_test_split(
    attributes, target, test_size=1000, stratify=target
)

In [513]:
numeric_columns = attributes_train[['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'CabinNum', 'total_spend']]
categorical_columns = attributes_train[['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']]

In [514]:
numeric_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', MinMaxScaler())
])

In [515]:
categorical_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(sparse_output=False)),
    ('scale', MinMaxScaler())
])

In [516]:
numeric_pipeline.fit(numeric_columns, target)

,steps,"[('impute', ...), ('scale', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,feature_range,"(0, ...)"


In [517]:
column_transformer = ColumnTransformer([
    ('numeric', numeric_pipeline, ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'CabinNum']),
    ('categorical', categorical_pipeline, ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side'])
], remainder='passthrough')

In [518]:
column_transformer.fit(attributes)

,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [519]:
base_pipeline = Pipeline([
    ('transform', column_transformer),
    ('model', RandomForestClassifier())
])

In [520]:
grid_search = GridSearchCV(
    estimator=base_pipeline,
    param_grid={
        'model__n_estimators': [100, 200, 500, 700, 1000]
    },
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

In [521]:
grid_search.fit(attributes_train, target_train)

Fitting 3 folds for each of 5 candidates, totalling 15 fits


,estimator,Pipeline(step...lassifier())])
,param_grid,"{'model__n_estimators': [100, 200, ...]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('numeric', ...), ('categorical', ...)]"


In [522]:
best_model = grid_search.best_estimator_
best_model

,steps,"[('transform', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [523]:
predictions = best_model.predict(attributes_test)

In [524]:
print(accuracy_score(target_test, predictions))

print(classification_report(target_test, predictions))

0.818
              precision    recall  f1-score   support

       False       0.80      0.85      0.82       496
        True       0.84      0.79      0.81       504

    accuracy                           0.82      1000
   macro avg       0.82      0.82      0.82      1000
weighted avg       0.82      0.82      0.82      1000



In [525]:
kaggle_test = pd.read_csv('data/test.csv')

In [526]:
kaggle_test['total_spend'] = (
    kaggle_test['RoomService'] +
    kaggle_test['FoodCourt'] +
    kaggle_test['ShoppingMall'] +
    kaggle_test['Spa'] +
    kaggle_test['VRDeck']
)

In [527]:
kaggle_test = kaggle_test.drop(columns=['Name'])
kaggle_test[['Deck', 'CabinNum', 'Side']] = (kaggle_test['Cabin'].str.split('/', expand=True))
kaggle_test = kaggle_test.drop(columns=['Cabin'])

In [528]:
test_ids = kaggle_test['PassengerId']

In [529]:
X_kaggle = kaggle_test.drop(columns=['PassengerId'])

In [530]:
predictions = best_model.predict(X_kaggle)

In [531]:
submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Transported': predictions.astype(bool)
})

In [532]:
submission.to_csv('submission2.csv', index=False)

### Problem 2. Dummy model(s)
Train two dummy models (use `most_frequent` and one other strategy of your choice - you may even want to write one yourself).

Do whatever you have to in order to make the model train, i.e., not throw an exception. You're allowed to disregard some columns for now.

Run your evaluation stats that you already described in Problem 1 and discuss the results.

Use 5-fold cross-validation. You might want to check out the [sklearn docs](https://scikit-learn.org/stable/modules/cross_validation.html) for more info and better cross-validation strategies. Also think about your testing set(s) - ask yourself what you're testing, why, and how. It's common to have more than one testing set for a model.

In [533]:
# Write your code here

### Problem 3. Data pipeline
Exclude the difficult columns `PassengerId`, `Cabin` and `Name`.

The most basic pipeline transforms all numeric, and all categorical features. You're free to choose your own transformation strategies, but I suggest imputation (with median) + Z-score for numeric; imputation (with mode) + one-hot encoding for categorical variables. Put them both in a `ColumnTransformer` and add a `LogisticRegression` after it, in a pipeline.

**Note:** Pipelines can be (and often are) nested inside each other.

Evaluate the data pipeline using your process from exercises 1 and 2.

In [534]:
# Write your code here

### Problem 4. Learning curve
A good method to see how more data affects the model is the `learning curve`. Typically more data should mean a better model, right?

Wrong. If the data is too noisy, the performance will drop. Also, if the model starts overfitting, you'll have a better idea of when and how it did so.

Look at the `sklearn` docs for more information. Interpret your results. You may want to answer questions like:
* Does the model seem to have high bias, high variance, both, or neither?
* Does validation performance improve, and to what extent, as we add more data?
* What is the most important next step: more data, more features, or a different (more complex) model?

In [535]:
# Write your code here

### Problem 5. Cabin features
From the `Cabin` column you can extract the deck, position (cabin number), and side.

I'd also add an indicator (boolean feature) of whether the cabin is missing, since this may be significant. If I merely impute it, I may lose some important information.

Now, build a `FeatureUnion` of the previous transformer (in problem 3) and this one. After extracting the parts impute (or create indicators), encode and / or scale - just like before.

In [536]:
# Write your code here

After you're done, compare both models. Be **extremely** careful not only to follow the initial process, but also to not accidentally leak data from one model to another. Pay very close attention on what data (and how many examples) you're training and evaluating.

You can use a very simple metric, like `score_with_cabin - score_without_cabin`. Document and discuss your findings.

**Hint:** Use the exact same folds as you did before. If you've done problem 3 correctly, this should be trivial. If not, go back and fix it.

**Hint 2:** Sometimes a feature is not useful because it was preprocessed poorly. Try your best to defend why you're choosing a preprocessing strategy, and make it reproducible.

In [537]:
# Write your code here

### Problem 6. Other features
If you want a more complete model, process `PassengerGroup` and other info (`Age`, `VIP`, `HomePlanet`, `Destination`) in a similar way. You may want to add other features, e.g. what's the size of each group? Which travellers were alone (group size = 1)? If not, it's permissible to ignore it for now; it's not going to teach you anything new. If you're trying to achieve a good score on the actual competition, you _will_ have to analyze them though.

Now, think about spending. Perform some feature engineering; e.g., what's the total spending and is it a useful feature? The answer will depend on a lot of factors, including what types of models you're training.

While you're here, consider these two things:
* What's the distribution of spending? What scaling might be better for it? Can we combine scalers?
* Is `CryoSleep` correlated to spending? A traveller who's sleeping will likely not spend anything. More gnerally, how do the spending amounts affect other features? Think about how wealthy a passenger may be, or what their position is (some cabins are more luxurious than others), or how many people they're travelling with.

In [538]:
# Write your code here

### Problem 7. A different model
Train and tune a logistic regression with your current pipeline. **Before** proceeding with a different model, prepare your protocol:
* Which hyperparameters are you choosing to optimize and how?
* Which hyperparameters are you choosing NOT to optimize and why?

Train and tune one more classifier of your choosing with the same pipeline, on the same data, using the same splits.

Compare the results fold by fold, just like in problem 4.

\* Optional: [Optuna](https://optuna.org/) is a library which allows you to do much better fine-tuning. It uses the optimization results so far to try and choose better hyperparameters to try out. Try to use it instead of `sklearn`'s default grid / randomized search.

In [539]:
# Write your code here

### Problem 8. Feature importances
If you've chosen trees or ensembles, you'll see they provide `feature_importances_`. This is - even by sklearn's docs - the **wrong** way to interpret them! Research what exactly is wrong (hint: it's not the code).

Think about what _important_ means and how you compute it: important for which model; what data subset; how do we deal with correlated features, etc.

Find a better way to compute feature importances. Apply it to both the logistic regression and your other model. Compare and discuss the results. 

What happens to feature importances for features that we split, e.g. we extracted 3 or 4 columns from the `Cabin` feature. In fact, what's the best order? Prepare `Cabin` for importances, then split into features; or run importances directly on `Deck`, `Position`, etc.? Describe and defend your answer. Consider using _feature groups_ for this task.

Finally, describe and discuss your results. Be careful how you phrase your results; especially not to confuse correlation with causation!

In [540]:
# Write your code here

### Problem 9. Error analysis
Choose your best model. Describe how and why you chose it.

A good model (even after considering everything we've said so far) isn't necesasrily the top scoring one. Other factors, such as robustness to outliers or inference speed are also important. Often, good interpretability (and model simplicity) are also important constraints.

Don't forget to follow the protocol from Exercise 1. Show more metrics; if needed test a few hypotheses about the data or the model performance as well.

In [541]:
# Write your code here

### Problem 10. More error analysis
Aggregations are not enough! Look at the data closely and try to find patterns in the model behavior. Some questions you might want to ask and answer are:
* Where does the model make the most mistakes?
* Are false positives and false negatives different?
* Are some passenger groups (e.g., older, VIP, sleeping, spending less) harder to classify?
* Are there cases where the model is confidently wrong? How do we measure this and what can we do about it?
* Do the errors suggest a systematic bias, e.g., a missing feature?
* Do the errors suggest a limitation of the data itself?
* What would you try next?
* When would you stop trying? What is a good measure of "good enough" performance?

Now write a final report.

In [542]:
# Write your code here

### * Problem 11. Full ablation study
Do what you did in Problem 5, but now apply it to all feature groups. Include feature importances, don't just look at scores.

Document and discuss your results. I find it most useful to prepare a table for different models, their hyperparameters, the procedure (e.g., has cabin? has spending?), and CV results.

In [543]:
# Write your code here